In [1]:
import google.generativeai as genai
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from glob import glob
import os
import json
from tqdm import tqdm

In [2]:
prompt_type = 'zeroshot'
dataset = 'type1'
image_type = 'combined'
model_name = "gemini_1.5_pro"

In [3]:
genai.configure(api_key= '<api_key>') #admusic key

In [4]:
generation_config = {
  "temperature": 1,
  "max_output_tokens": 10000,
}
safety_settings = [
  {
    "category": "HARM_CATEGORY_HARASSMENT",
    "threshold": "BLOCK_NONE"
  },
  {
    "category": "HARM_CATEGORY_HATE_SPEECH",
    "threshold": "BLOCK_NONE"
  },
  {
    "category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
    "threshold": "BLOCK_NONE"
  },
  {
    "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
    "threshold": "BLOCK_NONE"
  }
]
model = genai.GenerativeModel('gemini-1.5-pro-002', safety_settings=safety_settings, generation_config=generation_config)

def get_results(queries, max_workers=40):
    with ThreadPoolExecutor() as executor:
        executor._max_workers = max_workers
        results = list(executor.map(generate_content, queries))
    return results

def generate_content(query):
    try:
        resp = model.generate_content(query)
        print(".", end="")
        return resp.text
    except Exception as e:
        print(query, e)
        return 'Error by gemini'

In [5]:
if prompt_type == 'zeroshotcot':
    prompt = """Your task is the answer the question based on the given {img_word}. Your final answer to the question should strictly be in the format - "Final Answer:" <final_answer>.\nLet's work this out in a step by step way to be sure we have the right answer.\n\nQuestion: {question}"""
elif prompt_type == 'zeroshot':
    prompt = """Your task is the answer the question based on the given {img_word}. Your final answer to the question should strictly be in the format - "Final Answer:" <final_answer>.\n\nQuestion: {question}"""

In [6]:
index_to_image = {}
image_base_path = f'../{dataset}/{image_type}/'
if image_type == 'simple':
    img_word = 'images'
else:
    img_word = 'image'
    
def get_message(image_index, prompt, question):
    query = []
    for image in index_to_image[image_index]:
        image_path = f'{image_base_path}{image}'
        image = Image.open(image_path).convert('RGB')
        query.append(image)
        
    query.append(prompt.format(img_word=img_word, question=question))
    return query

def get_queries(qas):
    image_indices = qas['image_index'].values.astype(int)
    questions = qas['question'].values
    answers = qas['answer'].values
    

    all_images = os.listdir(image_base_path)

    if dataset == 'type1':
        prefix = 'multichart_'
        if image_type == 'original' or image_type == 'simple':
            sep = '_'
        else:
            sep = '.'

    for index in set(image_indices):
        for image in all_images:
            if image.startswith(f'{prefix}{index}{sep}'):
                if index not in index_to_image:
                    index_to_image[index] = []
                index_to_image[index].append(image)
    
    queries = []
    for i in tqdm(range(len(image_indices))):
        queries.append(get_message(image_indices[i], prompt, questions[i]))
        
    return questions, answers, queries
                
def get_message(image_index, prompt, question):
    query = []
    for image in index_to_image[image_index]:
        image_path = f'{image_base_path}{image}'
        image = Image.open(image_path).convert('RGB')
        query.append(image)
        
    query.append(prompt.format(img_word=img_word, question=question))
    return query


In [7]:
df = pd.read_json(f'../{dataset}/qa.json')
questions, answers, queries = get_queries(df)

  2%|▏         | 65/2809 [00:00<00:19, 138.65it/s]

100%|██████████| 2809/2809 [00:14<00:00, 189.77it/s]


In [8]:
results = get_results(queries, max_workers=1)

.

In [9]:
results_output = []
for i in range(len(questions)):
    results_output.append({
        'question_id': i,
        'question': questions[i],
        'answer': answers[i],
        'response': results[i]
    })
    
model_responses_df = pd.DataFrame(results_output)
model_responses_df.to_json(f'../model_responses/{dataset}/{model_name}_{image_type}_{prompt_type}.json', orient='records', indent=4)